# Classificação de Raças de Cachorros

Este notebook foi adaptado para a apresentação de Inteligência Artificial. Utiliza TensorFlow e TensorFlow Hub para treinar um modelo capaz de identificar raças de cachorros a partir de imagens.

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Furlovers/apresentacao-ia/blob/main/DogBreeds_Apresentacao.ipynb)

**Como rodar:** *Ambiente de execução → Executar tudo*. A primeira célula instala o que faltar e baixa os dados (cerca de 345 MB, só na primeira vez). No Colab, use um ambiente com GPU (T4) para o treino ser rápido.

In [ ]:
# 0. Setup: instala o que faltar e baixa os dados (só na primeira vez)
import os, sys, subprocess, importlib.util, urllib.request, zipfile

# Precisa vir antes de importar o TensorFlow, para o TF Hub usar o Keras 2 (tf_keras)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

EM_COLAB = "google.colab" in sys.modules
BASE_PATH = "/content/dados" if EM_COLAB else os.path.abspath("dados")
URL_DADOS = "https://github.com/Furlovers/apresentacao-ia/releases/download/v1.0-dados"


def instalar(*pacotes):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pacotes])


if importlib.util.find_spec("tensorflow") is None:
    # tf_keras fixa a versão do TensorFlow; instalando juntos, o pip escolhe um par compatível
    instalar("tensorflow", "tf_keras", "tensorflow_hub")
import tensorflow as tf

if importlib.util.find_spec("tf_keras") is None:
    versao_tf = ".".join(tf.__version__.split(".")[:2])
    instalar(f"tf_keras~={versao_tf}.0")
if importlib.util.find_spec("tensorflow_hub") is None:
    instalar("tensorflow_hub")
# O tensorflow_hub ainda importa pkg_resources, que saiu do setuptools a partir da versão 81
if importlib.util.find_spec("pkg_resources") is None:
    instalar("setuptools<81")
importlib.invalidate_caches()


def baixar(nome):
    destino = os.path.join(BASE_PATH, nome)
    print(f"Baixando {nome}...")
    urllib.request.urlretrieve(f"{URL_DADOS}/{nome}", destino)
    return destino


os.makedirs(BASE_PATH, exist_ok=True)
if not os.path.exists(os.path.join(BASE_PATH, "labels.csv")):
    baixar("labels.csv")

pasta_imagens = os.path.join(BASE_PATH, "train")
if not os.path.isdir(pasta_imagens) or len(os.listdir(pasta_imagens)) < 10222:
    zip_treino = baixar("train.zip")
    print("Extraindo as imagens...")
    with zipfile.ZipFile(zip_treino) as z:
        z.extractall(BASE_PATH)
    os.remove(zip_treino)

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "nenhuma")
print("Dados em", BASE_PATH, "|", len(os.listdir(pasta_imagens)), "imagens")

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

if EM_COLAB:
    from google.colab import files

## 1. Carregamento dos Dados
Visualização resumida das raças disponíveis no dataset.

In [ ]:
labels_csv = pd.read_csv(f'{BASE_PATH}/labels.csv')
print("Total de imagens:", len(labels_csv))
print("Total de raças únicas:", len(labels_csv['breed'].unique()))

labels_csv['breed'].value_counts().head(15).plot.bar(figsize=(12, 6), title='Top 15 Raças')
plt.show()

## 2. Pré-processamento e Separação de Classes
Preparação dos tensores de imagem e labels para treino e validação.

In [ ]:
# Criação de caminhos absolutos e arrays booleanos para as classes
filenames = [f'{BASE_PATH}/train/' + fname + '.jpg' for fname in labels_csv['id']]
unique_breeds = np.unique(labels_csv['breed'])
boolean_labels = [label == unique_breeds for label in labels_csv['breed']]

IMG_SIZE = 224


# Pre Processamento:

def process_image(image_path, img_size = IMG_SIZE):
  """
  Recebe o caminho da um arquivo de imagem e retorna um tensor.
  """

  # Le o arquivo:
  image = tf.io.read_file(image_path)

  # Conversão para um tensor com 3 canaispara cores
  image = tf.image.decode_jpeg(image, channels = 3)

  # Normalização do canal de cores (0-255) -> (0-1)
  image = tf.image.convert_image_dtype(image, tf.float32)

  # Correção do tamanho da imagem
  image = tf.image.resize(image, size = [IMG_SIZE, IMG_SIZE])

  return image

# Função para retornar a tupla (imagem, label)
def get_image_label(image_path, label):

    """
    Recebe  uuma imagem e seu rótulo, retornando a tupla (imagem, rótulo)
    """
    image = process_image(image_path)

    return image, label

# Definição do tamanho da Batch para treinamento:

BATCH_SIZE = 32

# Função para converter em batches:
def create_data_batches(X, y = None, batch_size = BATCH_SIZE, valid_data = False, test_data = False):

  """
  Cria as batches para treinamento
  """

  #No caso de dados de teste, não haverão labels
  if test_data:

    print("Criando batches de teste...\n")
    data = tf.data.Dataset.from_tensor_slices(tf.constant(X)) # Only filepaths, no labels
    data_batch = data.map(process_image).batch(BATCH_SIZE)

    return data_batch

  # Se for um dataset de validação, não há necessidade de embaralhar os dados:
  elif valid_data:

    print("Criando batches de validação...\n")
    data = tf.data.Dataset.from_tensor_slices((tf.constant(X), # Caminhos
                                              tf.constant(y))) # Labels

    data_batch = data.map(get_image_label).batch(BATCH_SIZE)

    return data_batch

  else:

    print("Criando batches de treinamento...\n")

    # Criando as tuplas
    data = tf.data.Dataset.from_tensor_slices((tf.constant(X), # Caminohs
                                              tf.constant(y))) # Labels

    # Embaralhando dados
    data = data.shuffle(buffer_size = len(X))

    # Mapeando as tuplas
    data = data.map(get_image_label)

    # Criação dads babtches
    data_batch = data.batch(BATCH_SIZE)

  return data_batch

## 3. Treinamento do Modelo (Transfer Learning)
Implementação de modelo preditivo via transferência de aprendizado com TensorFlow Hub.

In [ ]:
from sklearn.model_selection import train_test_split

# Definição dos tamanhos máximos dos sets de treino e teste
NUM_IMAGES = 1000

# Separação de entradas e labels:

X = filenames
y = boolean_labels

X_train, X_val, y_train, y_val = train_test_split(X[:NUM_IMAGES],
                                                  y[:NUM_IMAGES],
                                                  test_size=0.2,
                                                  random_state = 42)

# Verificação dos sets:
len(X_train), len(y_train), len(X_val), len(y_val)

# Criação das batches:
train_data = create_data_batches(X_train, y_train)

val_data = create_data_batches(X_val, y_val, valid_data=True)

In [ ]:
import tf_keras as keras

# MobileNetV2 (130/224) do TF Hub, que hoje fica no Kaggle Models (o tfhub.dev antigo pode não responder)
MODEL_URL = "https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/130-224-classification/2"
INPUT_SHAPE = [None, IMG_SIZE, IMG_SIZE, 3]
OUTPUT_SHAPE = len(unique_breeds)


def create_model(input_shape, output_shape, model_url):
    model = keras.Sequential([
        keras.layers.Input(shape=input_shape),
        hub.KerasLayer(model_url),
        keras.layers.Dense(units=output_shape, activation="softmax")
    ])
    model.compile(loss=keras.losses.CategoricalCrossentropy(),
                  optimizer=keras.optimizers.Adam(),
                  metrics=["accuracy"])
    return model


model = create_model((224, 224, 3), len(unique_breeds), MODEL_URL)
print("Iniciando o treinamento do modelo...")
model.fit(train_data, validation_data=val_data, epochs=5)

## 4. Inferência ao Vivo com Nova Imagem
Faça o upload de uma foto de um cachorro e passe pelo modelo treinado para prever a raça em tempo real.

No Colab, a célula abre o botão de upload. Fora do Colab, coloque o caminho de uma foto em `CAMINHO_IMAGEM`; se ficar `None`, o notebook usa uma foto da validação e mostra a raça correta para comparar.

In [ ]:
CAMINHO_IMAGEM = None  # fora do Colab: caminho de uma foto, ou None para usar uma da validação


def inferencia_live():
    raca_real = None
    if EM_COLAB:
        print("Envie sua imagem!")
        uploaded = files.upload()

        if not uploaded:
            print("Upload cancelado.")
            return

        img_path = list(uploaded.keys())[0]
    elif CAMINHO_IMAGEM:
        img_path = CAMINHO_IMAGEM
    else:
        i = np.random.randint(len(X_val))
        img_path = X_val[i]
        raca_real = unique_breeds[np.argmax(y_val[i])]

    display(Image(img_path, width=400))

    # Processando a imagem para o modelo
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, size=[224, 224])
    img = tf.expand_dims(img, axis=0)

    # Previsão
    pred = model.predict(img)
    label_index = np.argmax(pred[0])
    raca_prevista = unique_breeds[label_index]
    print(f"\n>>> A predição do modelo é a raça: {raca_prevista.upper()} <<<")
    if raca_real is not None:
        print(f"    Raça correta: {raca_real.upper()}")


inferencia_live()